# Blending of images 
> Semiconductor-specific blending. 

In [ ]:
#| default_exp image_gen.advanced_blending

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| export
import sys
from pathlib import Path


In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv


In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/Users/goni/workspace/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/Users/goni/workspace/projects/git_data/new_test/nbs'

In [ ]:
#| export
import cv2
import numpy as np
import os
import hashlib
from pathlib import Path
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import random
from scipy import ndimage
from skimage import exposure, filters, morphology, segmentation
from skimage.restoration import denoise_wavelet
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
import pickle

from fastcore.script import *
from fastcore.all import *

In [ ]:
#| export
class SemiconductorImageSynthesizer:
    def __init__(self, base_path: str):
        self.base_path = Path(base_path)
        self.fg_bank_path = self.base_path / "foreground_bank"
        self.bg_bank_path = self.base_path / "background_bank"
        self.synthetic_path = self.base_path / "synthetic_images"
        self.masks_path = self.base_path / "synthetic_masks"
        
        # Create directories
        for path in [self.fg_bank_path, self.bg_bank_path, self.synthetic_path, self.masks_path]:
            path.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
@patch
def extract_foreground_background(
    self:SemiconductorImageSynthesizer, 
    image_path: str, 
    mask_path: str
    ):
    """Extract foreground and background from segmented images"""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
    if image is None or mask is None:
        return None, None
        
    # Normalize mask to binary
    _, mask_binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        
    # Extract foreground with alpha channel for transparency
    foreground = np.zeros((image.shape[0], image.shape[1], 2), dtype=np.uint8)
    foreground[:, :, 0] = image
    foreground[:, :, 1] = mask_binary  # Alpha channel
        
    # Extract background (invert mask)
    bg_mask = cv2.bitwise_not(mask_binary)
    background = np.zeros((image.shape[0], image.shape[1], 2), dtype=np.uint8)
    background[:, :, 0] = image
    background[:, :, 1] = bg_mask
        
    return foreground, background

In [ ]:
#| export
@patch
def compute_image_hash(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray
    ):
    """Compute perceptual hash for deduplication"""
    # Resize to standard size for consistent hashing
    # Handle both grayscale and color images - extract first channel if color, use as-is if grayscale
    resized = cv2.resize(image if len(image.shape) == 2 else image[:,:,0], (64, 64))
        
    # Apply Gaussian blur to reduce noise sensitivity
    blurred = cv2.GaussianBlur(resized, (3, 3), 0)
        
    # Compute DCT-based hash
    dct = cv2.dct(np.float32(blurred))
    dct_low = dct[:8, :8]
    median = np.median(dct_low)
    hash_bits = (dct_low > median).flatten()
        
    return hashlib.md5(hash_bits.tobytes()).hexdigest()

In [ ]:
#| export
@patch
def compute_feature_vector(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray) -> np.ndarray:
    """Compute feature vector for similarity comparison"""
    img = image if len(image.shape) == 2 else image[:,:,0]
    img_resized = cv2.resize(img, (128, 128))
        
    # Multiple feature descriptors
    features = []
        
    # Histogram features
    hist = cv2.calcHist([img_resized], [0], None, [32], [0, 256])
    features.extend(hist.flatten())
        
    # Texture features using LBP-like approach
    sobel_x = cv2.Sobel(img_resized, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img_resized, cv2.CV_64F, 0, 1, ksize=3)
    gradient_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    texture_hist = np.histogram(gradient_mag, bins=16)[0]
    features.extend(texture_hist)
        
    # Statistical moments
    features.extend([np.mean(img_resized), np.std(img_resized), 
                    np.percentile(img_resized, 25), np.percentile(img_resized, 75)])
        
    return np.array(features, dtype=np.float32)
   

In [ ]:
#| export
@patch
def deduplicate_images(
    self:SemiconductorImageSynthesizer, 
    images: List[np.ndarray], 
    threshold: float = 0.95
    ) -> List[int]:
    """Remove duplicate images using perceptual hashing and feature similarity"""
    if not images:
        return []
        
    # Compute hashes and features
    hashes = []
    features = []
        
    for img in images:
        hashes.append(self.compute_image_hash(img))
        features.append(self.compute_feature_vector(img))
        
    # Remove exact hash duplicates
    unique_indices = []
    seen_hashes = set()
        
    for i, hash_val in enumerate(hashes):
        if hash_val not in seen_hashes:
            unique_indices.append(i)
            seen_hashes.add(hash_val)
        
    if len(unique_indices) <= 1:
        return unique_indices
        
    # Remove near-duplicates using feature similarity
    unique_features = [features[i] for i in unique_indices]
    similarity_matrix = cosine_similarity(unique_features)
        
    final_indices = []
    used = set()
        
    for i in range(len(unique_indices)):
        if i in used:
            continue
                
        final_indices.append(unique_indices[i])
            
        # Mark similar images as used
        for j in range(i + 1, len(unique_indices)):
            if similarity_matrix[i, j] > threshold:
                used.add(j)
        
    return final_indices
    



In [ ]:
#| export
@patch
def advanced_poisson_blending(
    self:SemiconductorImageSynthesizer, 
    fg_img: np.ndarray, # foreground image
    bg_img: np.ndarray, # background image
    fg_mask: np.ndarray, # foreground mask
    offset: Tuple[int, int] = (0, 0) # offset for the foreground image
    ) -> np.ndarray:
        """Advanced Poisson blending with gradient domain processing"""
        
        # Ensure images are same size
        h, w = bg_img.shape[:2]
        fg_resized = cv2.resize(fg_img, (w, h))
        mask_resized = cv2.resize(fg_mask, (w, h))
        
        # Ensure mask is binary
        _, mask_resized = cv2.threshold(mask_resized, 127, 255, cv2.THRESH_BINARY)
        
        # Create smooth mask with feathering (gradual edge transition to avoid harsh boundaries)
        # Feathering creates a soft gradient at mask edges for seamless blending
        kernel_size = max(5, min(w, h) // 50)  # Adaptive kernel size based on image dimensions
        if kernel_size % 2 == 0:
            kernel_size += 1  # Ensure odd kernel size for symmetric blur
        
        smooth_mask = cv2.GaussianBlur(mask_resized, (kernel_size, kernel_size), 0)
        smooth_mask = smooth_mask.astype(np.float32) / 255.0
        
        # Multi-scale blending
        result = bg_img.astype(np.float32)

        # Create Laplacian pyramid for multi-scale blending
        # This breaks the image into different frequency bands (detail levels)
        # allowing us to blend fine details separately from coarse structure
        def build_laplacian_pyramid(img, levels=4):
            # Step 1: Build Gaussian pyramid (progressively smaller, blurred versions)
            gaussian = [img.astype(np.float32)]
            for i in range(levels):
                gaussian.append(cv2.pyrDown(gaussian[i]))  # Downsample and blur
            
            # Step 2: Build Laplacian pyramid (difference between levels = edge/detail info)
            laplacian = []
            for i in range(levels):
                size = gaussian[i].shape[:2][::-1]  # (width, height) for pyrUp
                expanded = cv2.pyrUp(gaussian[i+1], dstsize=size)  # Upsample next level
                laplacian.append(gaussian[i] - expanded)  # Subtract = isolate details
            laplacian.append(gaussian[-1])  # Add the smallest (most blurred) level
            
            return laplacian
        
        def build_gaussian_pyramid(img, levels=4):
            pyramid = [img.astype(np.float32)]
            for i in range(levels):
                pyramid.append(cv2.pyrDown(pyramid[i]))
            return pyramid
        # Build pyramids
        levels = 4
        fg_laplacian = build_laplacian_pyramid(fg_resized, levels)
        bg_laplacian = build_laplacian_pyramid(bg_img, levels)
        mask_gaussian = build_gaussian_pyramid(smooth_mask, levels)
        
        # Blend in pyramid domain
        blended_laplacian = []
        for i in range(len(fg_laplacian)):
            if len(mask_gaussian[i].shape) != 2:
                print(f"mask_gaussian[i].shape: {mask_gaussian[i].shape}")
                mask_level = mask_gaussian[i][:, :, np.newaxis]
            else:
                mask_level = mask_gaussian[i]
            
            blended = fg_laplacian[i] * mask_level + bg_laplacian[i] * (1 - mask_level)
            blended_laplacian.append(blended)

        print(f"blended_laplacian done")
        
        # Reconstruct image
        result = blended_laplacian[-1]
        for i in range(len(blended_laplacian) - 2, -1, -1):
            size = blended_laplacian[i].shape[:2][::-1]
            result = cv2.pyrUp(result, dstsize=size) + blended_laplacian[i]
        
        return np.clip(result, 0, 255).astype(np.uint8)
        
       

In [ ]:
#| export
@patch
def seamless_histogram_matching(
    self:SemiconductorImageSynthesizer, 
    source: np.ndarray, # source image
    template: np.ndarray, # template image
    mask: np.ndarray # mask
    ) -> np.ndarray:
    """Match histogram in masked region for seamless blending"""
    # Ensure all inputs have the same dimensions
    h, w = template.shape[:2]

    # Resize source and mask to match template dimensions if needed
    if source.shape[:2] != (h, w):
        source = cv2.resize(source, (w, h))
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h))
    
    result = source.copy().astype(np.float32)
        
    # Extract masked regions
    mask_bool = mask > 127
    if not np.any(mask_bool):
        return source.astype(np.uint8)

    source_masked = source[mask_bool]
        
    # Find template region around the mask for better matching
    # Use smaller kernel to avoid dimension issues
    kernel_size = min(20, min(h, w) // 10)  # Adaptive kernel size
    if kernel_size < 3:
        kernel_size = 3
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    # Ensure dilated mask stays within image bounds
    dilated_mask = cv2.dilate(mask, kernel, iterations=1)
    
    # Make sure dilated_mask has same dimensions as template
    if dilated_mask.shape[:2] != (h, w):
        dilated_mask = cv2.resize(dilated_mask, (w, h))
    
    # Ensure mask also has same dimensions before XOR operation
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h))
    
    template_region = cv2.bitwise_xor(dilated_mask, mask)
    
    # Ensure template_region has same dimensions as template
    if template_region.shape[:2] != (h, w):
        template_region = cv2.resize(template_region, (w, h))
    
    template_bool = template_region > 127
        
    # Verify dimensions match before boolean indexing
    if template_bool.shape[:2] != template.shape[:2]:
        print(f"Dimension mismatch: template_bool {template_bool.shape} vs template {template.shape}")
        return source.astype(np.uint8)
        
    if not np.any(template_bool):
        template_masked = template.flatten()
    else:
        template_masked = template[template_bool]
        
    # Ensure we have enough data for histogram matching
    if len(source_masked) == 0 or len(template_masked) == 0:
        return source.astype(np.uint8)
        
    # Match histograms
    try:
        matched = exposure.match_histograms(source_masked, template_masked)
        result[mask_bool] = matched
    except Exception as e:
        print(f"Warning: Histogram matching failed: {e}. Returning original source.")
        return source.astype(np.uint8)
        
    return result.astype(np.uint8)

In [ ]:
#| export
@patch
def noise_matching(
    self:SemiconductorImageSynthesizer, 
    synthetic: np.ndarray, # synthetic image
    reference: np.ndarray # reference image
    ) -> np.ndarray:
    """Match noise characteristics to reference image"""
    # Estimate noise in reference
    ref_denoised = denoise_wavelet(reference, method='BayesShrink', mode='soft')
    ref_noise = reference.astype(np.float32) - ref_denoised.astype(np.float32)
    noise_std = np.std(ref_noise)
        
    # Add similar noise to synthetic image
    synthetic_float = synthetic.astype(np.float32)
    noise = np.random.normal(0, noise_std * 0.3, synthetic.shape)
        
    # Apply noise with edge preservation
    edges = cv2.Canny(synthetic, 50, 150)
    edge_mask = cv2.dilate(edges, np.ones((3,3), np.uint8))
        
    result = synthetic_float.copy()
    non_edge_mask = edge_mask == 0
    result[non_edge_mask] += noise[non_edge_mask]
        
    return np.clip(result, 0, 255).astype(np.uint8)
    

In [ ]:
#| export
@patch
def create_synthetic_image(
    self:SemiconductorImageSynthesizer, 
    fg_img: np.ndarray, # foreground image
    bg_img: np.ndarray, # background image
    apply_defects: bool = True # whether to apply realistic pin defects
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Create high-quality synthetic image with undetectable blending and realistic pin defects"""
    
    try:
        # Apply realistic pin defects BEFORE blending for maximum realism
        if apply_defects:
            fg_img_with_defects = self.apply_realistic_pin_defects(fg_img)
            print(f"Applied realistic pin defects")
        else:
            fg_img_with_defects = fg_img
        
        # Extract channels from potentially modified foreground
        fg_image = fg_img_with_defects[:, :, 0]
        fg_mask = fg_img_with_defects[:, :, 1]
        bg_image = bg_img[:, :, 0]
        bg_mask = bg_img[:, :, 1]
        
        # Ensure all images have compatible dimensions by resizing to background size
        h, w = bg_image.shape[:2]
        if fg_image.shape[:2] != (h, w):
            fg_image = cv2.resize(fg_image, (w, h))
        if fg_mask.shape[:2] != (h, w):
            fg_mask = cv2.resize(fg_mask, (w, h))
        
        # Random transformations for variety (reduced to avoid affecting defects too much)
        if random.random() > 0.7:  # Reduced probability to preserve defect realism
            # Small random rotation
            angle = random.uniform(-8, 8)  # Reduced angle range
            center = (fg_image.shape[1]//2, fg_image.shape[0]//2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            fg_image = cv2.warpAffine(fg_image, M, (fg_image.shape[1], fg_image.shape[0]))
            fg_mask = cv2.warpAffine(fg_mask, M, (fg_mask.shape[1], fg_mask.shape[0]))
            
        if random.random() > 0.8:  # Reduced probability 
            # Small random scaling
            scale = random.uniform(0.9, 1.1)  # Reduced scale range
            new_size = (int(fg_image.shape[1] * scale), int(fg_image.shape[0] * scale))
            fg_image = cv2.resize(fg_image, new_size)
            fg_mask = cv2.resize(fg_mask, new_size)
            
        # After transformations, ensure dimensions match background again
        if fg_image.shape[:2] != (h, w):
            fg_image = cv2.resize(fg_image, (w, h))
        if fg_mask.shape[:2] != (h, w):
            fg_mask = cv2.resize(fg_mask, (w, h))

        # Histogram matching for illumination consistency
        fg_matched = self.seamless_histogram_matching(fg_image, bg_image, fg_mask)

        # Advanced Poisson blending
        synthetic = self.advanced_poisson_blending(fg_matched, bg_image, fg_mask)
            
        # Noise matching for realism
        synthetic = self.noise_matching(synthetic, bg_image)
            
        # Create final mask
        final_mask = fg_mask.copy()
            
        # Post-processing for undetectable blending
        # More subtle processing to preserve defect realism
        synthetic = cv2.bilateralFilter(synthetic, 3, 15, 15)  # Reduced filtering
            
        # Very subtle sharpening to avoid over-processing defects
        kernel = np.array([[-0.5,-0.5,-0.5], [-0.5,5,-0.5], [-0.5,-0.5,-0.5]])
        sharpened = cv2.filter2D(synthetic, -1, kernel)
        synthetic = cv2.addWeighted(synthetic, 0.95, sharpened, 0.05, 0)  # Reduced sharpening
            
        return synthetic, final_mask
    
    except Exception as e:
        print(f"Error in create_synthetic_image: {e}")
        # Return a simple blend as fallback
        h, w = bg_img.shape[:2]
        fg_resized = cv2.resize(fg_img[:, :, 0], (w, h))
        mask_resized = cv2.resize(fg_img[:, :, 1], (w, h))
        
        # Simple alpha blending as fallback
        alpha = mask_resized.astype(np.float32) / 255.0
        result = (fg_resized * alpha + bg_img[:, :, 0] * (1 - alpha)).astype(np.uint8)
        
        return result, mask_resized

In [ ]:
#| export
@patch
def generate_dirt_contamination(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    intensity: float = 0.3 # contamination intensity (0.1-0.8)
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Add realistic dirt/contamination to pin surface"""
    
    result = image.copy().astype(np.float32)
    contaminated_mask = mask.copy()
    
    # Create contamination patterns
    h, w = image.shape[:2]
    
    # Generate multiple contamination spots
    num_spots = random.randint(1, 4)
    
    for _ in range(num_spots):
        # Random contamination center within pin area
        pin_coords = np.where(mask > 127)
        if len(pin_coords[0]) == 0:
            continue
            
        idx = random.randint(0, len(pin_coords[0]) - 1)
        center_y, center_x = pin_coords[0][idx], pin_coords[1][idx]
        
        # Create contamination blob with irregular shape
        blob_size = random.randint(3, min(h, w) // 8)
        
        # Use Perlin-like noise for organic contamination shape
        y_coords, x_coords = np.ogrid[:h, :w]
        
        # Multiple frequency components for realistic contamination
        contamination = np.zeros((h, w), dtype=np.float32)
        
        for freq_scale in [1.0, 0.5, 0.25]:
            freq = blob_size * freq_scale
            noise_y = (y_coords - center_y) / freq
            noise_x = (x_coords - center_x) / freq
            
            # Create organic blob shape using multiple sine waves
            blob = np.exp(-(noise_y**2 + noise_x**2) / 2)
            blob *= np.sin(noise_x * 3.14159) * np.cos(noise_y * 3.14159)
            blob = np.maximum(0, blob)
            
            contamination += blob * freq_scale
        
        # Normalize and apply threshold for contamination area
        contamination = np.clip(contamination, 0, 1)
        contamination_mask = contamination > 0.2
        
        # Only apply within pin area
        contamination_mask = contamination_mask & (mask > 127)
        
        if np.any(contamination_mask):
            # Darken contaminated areas (dirt makes surface darker)
            contamination_strength = intensity * contamination[contamination_mask]
            result[contamination_mask] *= (1 - contamination_strength * 0.6)
            
            # Add slight texture variation
            texture_noise = np.random.normal(0, 5, np.sum(contamination_mask))
            result[contamination_mask] += texture_noise
            
            # Update mask to include contamination (slightly expand pin area)
            kernel = np.ones((3, 3), np.uint8)
            expanded_contamination = cv2.dilate(contamination_mask.astype(np.uint8) * 255, kernel)
            contaminated_mask = np.maximum(contaminated_mask, expanded_contamination)
    
    return np.clip(result, 0, 255).astype(np.uint8), contaminated_mask


In [ ]:
#| export
@patch
def generate_partial_shadow(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    shadow_intensity: float = 0.4 # shadow darkness (0.1-0.7)
    ) -> np.ndarray:
    """Add realistic partial shadows that obscure part of the pin"""
    
    result = image.copy().astype(np.float32)
    h, w = image.shape[:2]
    
    # Create shadow gradient - simulate lighting from one side
    shadow_direction = random.choice(['left', 'right', 'top', 'bottom', 'diagonal'])
    
    if shadow_direction == 'left':
        # Shadow from left side
        x_coords = np.arange(w)
        shadow_gradient = 1 - (x_coords / w) * shadow_intensity
        shadow_map = np.tile(shadow_gradient, (h, 1))
        
    elif shadow_direction == 'right':
        # Shadow from right side  
        x_coords = np.arange(w)
        shadow_gradient = 1 - ((w - x_coords) / w) * shadow_intensity
        shadow_map = np.tile(shadow_gradient, (h, 1))
        
    elif shadow_direction == 'top':
        # Shadow from top
        y_coords = np.arange(h)
        shadow_gradient = 1 - (y_coords / h) * shadow_intensity
        shadow_map = np.tile(shadow_gradient.reshape(-1, 1), (1, w))
        
    elif shadow_direction == 'bottom':
        # Shadow from bottom
        y_coords = np.arange(h)
        shadow_gradient = 1 - ((h - y_coords) / h) * shadow_intensity
        shadow_map = np.tile(shadow_gradient.reshape(-1, 1), (1, w))
        
    else:  # diagonal
        # Diagonal shadow
        y_coords, x_coords = np.ogrid[:h, :w]
        center_y, center_x = h // 2, w // 2
        
        # Random diagonal direction
        angle = random.uniform(0, 2 * np.pi)
        shadow_direction_x = np.cos(angle)
        shadow_direction_y = np.sin(angle)
        
        # Project coordinates onto shadow direction
        projected = ((y_coords - center_y) * shadow_direction_y + 
                    (x_coords - center_x) * shadow_direction_x)
        
        # Normalize and create gradient
        proj_min, proj_max = projected.min(), projected.max()
        if proj_max > proj_min:
            projected_norm = (projected - proj_min) / (proj_max - proj_min)
            shadow_map = 1 - projected_norm * shadow_intensity
        else:
            shadow_map = np.ones((h, w))
    
    # Add some noise to shadow boundary for realism
    noise = np.random.normal(0, 0.05, (h, w))
    shadow_map += noise
    shadow_map = np.clip(shadow_map, 0.3, 1.0)  # Prevent complete darkness
    
    # Apply shadow only to pin area
    pin_area = mask > 127
    result[pin_area] *= shadow_map[pin_area]
    
    # Add subtle edge softening at shadow boundaries
    shadow_edges = cv2.Canny((shadow_map * 255).astype(np.uint8), 50, 150)
    shadow_edges = cv2.dilate(shadow_edges, np.ones((3, 3), np.uint8))
    edge_blur = cv2.GaussianBlur(result, (3, 3), 0)
    
    # Blend at edges for smooth transition
    edge_mask = (shadow_edges > 0) & pin_area
    result[edge_mask] = 0.7 * result[edge_mask] + 0.3 * edge_blur[edge_mask]
    
    return np.clip(result, 0, 255).astype(np.uint8)


In [ ]:
#| export
@patch
def generate_oxidation_corrosion(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    severity: float = 0.3 # oxidation severity (0.1-0.6)
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Add realistic oxidation/corrosion patterns to pin surface"""
    
    result = image.copy().astype(np.float32)
    corroded_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Create corrosion pattern - typically starts at edges and spreads inward
    pin_coords = np.where(mask > 127)
    if len(pin_coords[0]) == 0:
        return image, mask
    
    # Find pin edges
    kernel = np.ones((3, 3), np.uint8)
    pin_edges = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)
    edge_coords = np.where(pin_edges > 127)
    
    if len(edge_coords[0]) == 0:
        return image, mask
    
    # Generate multiple corrosion centers along edges
    num_corrosion_spots = random.randint(2, 6)
    
    for _ in range(num_corrosion_spots):
        # Pick random edge point as corrosion center
        idx = random.randint(0, len(edge_coords[0]) - 1)
        center_y, center_x = edge_coords[0][idx], edge_coords[1][idx]
        
        # Create corrosion spreading pattern
        y_coords, x_coords = np.ogrid[:h, :w]
        
        # Distance from corrosion center
        dist_from_center = np.sqrt((y_coords - center_y)**2 + (x_coords - center_x)**2)
        
        # Corrosion spreads with decreasing intensity
        max_spread = min(h, w) // 4 * severity
        corrosion_intensity = np.exp(-dist_from_center / max_spread)
        
        # Add randomness for organic corrosion pattern
        noise = np.random.random((h, w))
        corrosion_pattern = corrosion_intensity * noise
        
        # Threshold to create corrosion areas
        corrosion_mask = (corrosion_pattern > 0.3) & (mask > 127)
        
        if np.any(corrosion_mask):
            # Corrosion changes surface properties
            # Make surface darker and add texture
            corrosion_factor = corrosion_pattern[corrosion_mask] * severity
            
            # Darken corroded areas
            result[corrosion_mask] *= (1 - corrosion_factor * 0.4)
            
            # Add brownish tint (simulate oxidation)
            # In grayscale, this means slight brightness variation
            oxidation_variation = np.random.normal(0, 10, np.sum(corrosion_mask))
            result[corrosion_mask] += oxidation_variation * corrosion_factor
            
            # Add surface roughness
            roughness = np.random.normal(0, 8, np.sum(corrosion_mask))
            result[corrosion_mask] += roughness * corrosion_factor
            
            # Expand mask slightly to include corrosion
            kernel_small = np.ones((2, 2), np.uint8)
            expanded_corrosion = cv2.dilate(corrosion_mask.astype(np.uint8) * 255, kernel_small)
            corroded_mask = np.maximum(corroded_mask, expanded_corrosion)
    
    return np.clip(result, 0, 255).astype(np.uint8), corroded_mask


In [ ]:
#| export
@patch
def generate_flux_residue(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    residue_amount: float = 0.2 # amount of flux residue (0.1-0.5)
    ) -> Tuple[np.ndarray, np.ndarray]:
    """Add realistic flux residue patterns around pin"""
    
    result = image.copy().astype(np.float32)
    residue_mask = mask.copy()
    h, w = image.shape[:2]
    
    # Flux residue typically appears around pin base and edges
    kernel = np.ones((5, 5), np.uint8)
    pin_dilated = cv2.dilate(mask, kernel, iterations=2)
    residue_area = cv2.bitwise_xor(pin_dilated, mask)
    
    # Create organic residue patterns
    residue_coords = np.where(residue_area > 127)
    if len(residue_coords[0]) == 0:
        return image, mask
    
    # Generate residue blobs
    num_residue_areas = random.randint(3, 8)
    
    for _ in range(num_residue_areas):
        # Random residue center in residue area
        idx = random.randint(0, len(residue_coords[0]) - 1)
        center_y, center_x = residue_coords[0][idx], residue_coords[1][idx]
        
        # Create residue blob
        y_coords, x_coords = np.ogrid[:h, :w]
        blob_size = random.randint(3, min(h, w) // 6)
        
        # Organic blob shape
        dist = np.sqrt((y_coords - center_y)**2 + (x_coords - center_x)**2)
        blob = np.exp(-dist / blob_size)
        
        # Add irregularity
        angle_map = np.arctan2(y_coords - center_y, x_coords - center_x)
        irregularity = 1 + 0.3 * np.sin(angle_map * random.randint(3, 7))
        blob *= irregularity
        
        # Threshold and apply to residue area
        blob_mask = (blob > 0.3) & (residue_area > 127)
        
        if np.any(blob_mask):
            # Flux residue characteristics
            # Slightly darker/lighter than background with texture
            residue_intensity = blob[blob_mask] * residue_amount
            
            # Some residue is darker, some lighter
            if random.random() > 0.5:
                # Dark residue
                result[blob_mask] *= (1 - residue_intensity * 0.3)
            else:
                # Light residue
                result[blob_mask] *= (1 + residue_intensity * 0.2)
            
            # Add texture
            texture = np.random.normal(0, 5, np.sum(blob_mask))
            result[blob_mask] += texture * residue_intensity
            
            # Update mask to include residue
            residue_mask = np.maximum(residue_mask, blob_mask.astype(np.uint8) * 255)
    
    return np.clip(result, 0, 255).astype(np.uint8), residue_mask


In [ ]:
#| export
@patch
def generate_surface_scratches(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # input pin image
    mask: np.ndarray, # pin mask
    scratch_intensity: float = 0.3 # scratch visibility (0.1-0.6)
    ) -> np.ndarray:
    """Add realistic surface scratches to pin"""
    
    result = image.copy().astype(np.float32)
    h, w = image.shape[:2]
    
    # Generate random scratches
    num_scratches = random.randint(1, 4)
    
    for _ in range(num_scratches):
        # Random scratch parameters
        scratch_length = random.randint(min(h, w) // 4, min(h, w) // 2)
        scratch_width = random.randint(1, 3)
        
        # Random start point within pin area
        pin_coords = np.where(mask > 127)
        if len(pin_coords[0]) == 0:
            continue
            
        idx = random.randint(0, len(pin_coords[0]) - 1)
        start_y, start_x = pin_coords[0][idx], pin_coords[1][idx]
        
        # Random scratch direction
        angle = random.uniform(0, 2 * np.pi)
        dx = np.cos(angle)
        dy = np.sin(angle)
        
        # Create scratch line
        scratch_mask = np.zeros((h, w), dtype=np.uint8)
        
        for i in range(scratch_length):
            x = int(start_x + i * dx)
            y = int(start_y + i * dy)
            
            if 0 <= x < w and 0 <= y < h and mask[y, x] > 127:
                # Draw scratch with some width
                for dx_offset in range(-scratch_width, scratch_width + 1):
                    for dy_offset in range(-scratch_width, scratch_width + 1):
                        nx, ny = x + dx_offset, y + dy_offset
                        if 0 <= nx < w and 0 <= ny < h:
                            scratch_mask[ny, nx] = 255
        
        # Apply scratch effect
        scratch_coords = np.where(scratch_mask > 127)
        if len(scratch_coords[0]) > 0:
            # Scratches can be darker or lighter
            if random.random() > 0.5:
                # Dark scratch
                result[scratch_coords] *= (1 - scratch_intensity * 0.4)
            else:
                # Light scratch (reflection)
                result[scratch_coords] *= (1 + scratch_intensity * 0.3)
            
            # Add slight blur to scratch edges for realism
            kernel = np.ones((2, 2), np.uint8)
            scratch_edges = cv2.morphologyEx(scratch_mask, cv2.MORPH_GRADIENT, kernel)
            edge_coords = np.where(scratch_edges > 127)
            
            if len(edge_coords[0]) > 0:
                result[edge_coords] = 0.8 * result[edge_coords] + 0.2 * cv2.GaussianBlur(result, (3, 3), 0)[edge_coords]
    
    return np.clip(result, 0, 255).astype(np.uint8)


In [ ]:
#| export
@patch
def generate_defective_pin_variants(
    self:SemiconductorImageSynthesizer, 
    original_image: np.ndarray, # original pin image
    original_mask: np.ndarray, # original pin mask
    num_variants: int = 10, # number of defective variants to generate
    intensity_range: Tuple[float, float] = (0.4, 0.8) # defect intensity range (high intensity)
    ) -> List[Tuple[np.ndarray, np.ndarray, str]]:
    """Generate multiple defective variants with REALISTIC rectangular defects"""
    
    variants = []
    # Enhanced defect types with deep shadows that can completely hide pin parts
    defect_types = ['dirt', 'deep_shadow', 'oxidation', 'flux', 'scratches']
    
    for i in range(num_variants):
        # Create variant with random defect combination
        variant_image = original_image.copy()
        variant_mask = original_mask.copy()
        applied_defects = []
        
        # Randomly select 1-3 defect types for this variant
        num_defects = random.randint(1, 3)
        selected_defects = random.sample(defect_types, num_defects)
        
        # Apply each selected defect with REALISTIC effects constrained to rectangles
        for defect_type in selected_defects:
            # Use high intensity range for visible defects
            intensity = random.uniform(intensity_range[0], intensity_range[1])
            
            if defect_type == 'dirt':
                variant_image, variant_mask = self.generate_realistic_rectangular_dirt_contamination(
                    variant_image, variant_mask, contamination_level=intensity
                )
                applied_defects.append(f"realistic_dirt_{intensity:.2f}")
                
            elif defect_type == 'deep_shadow':
                # Use higher intensity for deep shadows that can hide pin parts
                shadow_intensity = random.uniform(0.7, 0.95)  # Very deep shadows
                variant_image, variant_mask = self.generate_realistic_rectangular_deep_shadow(
                    variant_image, variant_mask, shadow_intensity=shadow_intensity
                )
                applied_defects.append(f"deep_shadow_{shadow_intensity:.2f}")
                
            elif defect_type == 'oxidation':
                variant_image, variant_mask = self.generate_realistic_rectangular_oxidation_corrosion(
                    variant_image, variant_mask, severity=intensity
                )
                applied_defects.append(f"realistic_oxidation_{intensity:.2f}")
                
            elif defect_type == 'flux':
                variant_image, variant_mask = self.generate_realistic_rectangular_flux_residue(
                    variant_image, variant_mask, residue_amount=intensity
                )
                applied_defects.append(f"realistic_flux_{intensity:.2f}")
                
            elif defect_type == 'scratches':
                variant_image = self.generate_realistic_rectangular_surface_scratches(
                    variant_image, variant_mask, scratch_intensity=intensity
                )
                applied_defects.append(f"realistic_scratches_{intensity:.2f}")
        
        # Create description of applied defects
        defect_description = "_".join(applied_defects)
        
        variants.append((variant_image, variant_mask, defect_description))
    
    return variants


In [ ]:
#| export
@patch
def process_defect_augmentation_dataset(
    self:SemiconductorImageSynthesizer, 
    image_dir: str, # directory containing original pin images
    mask_dir: str, # directory containing corresponding masks
    variants_per_image: int = 15, # number of defective variants per original image
    intensity_range: Tuple[float, float] = (0.5, 0.9) # high intensity defects
    ):
    """Process dataset to generate defective pin variants from normal images"""
    
    print("🎯 Starting Defect Augmentation Process...")
    print(f"Target: {variants_per_image} defective variants per original image")
    print(f"Defect intensity range: {intensity_range[0]:.1f} - {intensity_range[1]:.1f}")
    
    # Get all image files
    image_files = list(Path(image_dir).glob("*.png")) + list(Path(image_dir).glob("*.jpg"))
    print(f"Found {len(image_files)} original images to process")
    
    total_generated = 0
    successful_images = 0
    
    for img_idx, img_file in enumerate(image_files):
        mask_file = Path(mask_dir) / img_file.name
        
        if not mask_file.exists():
            print(f"⚠️ Skipping {img_file.name} - no corresponding mask found")
            continue
            
        try:
            # Load original image and mask
            original_image = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            original_mask = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
            
            if original_image is None or original_mask is None:
                print(f"⚠️ Failed to load {img_file.name}")
                continue
            
            print(f"\n📸 Processing image {img_idx + 1}/{len(image_files)}: {img_file.name}")
            
            # Generate defective variants
            variants = self.generate_defective_pin_variants(
                original_image, original_mask, 
                num_variants=variants_per_image,
                intensity_range=intensity_range
            )
            
            # Save each variant
            base_name = img_file.stem
            for var_idx, (variant_img, variant_mask, defect_desc) in enumerate(variants):
                # Create descriptive filename
                variant_name = f"{base_name}_defect_{var_idx:02d}_{defect_desc}.png"
                mask_name = f"{base_name}_defect_{var_idx:02d}_{defect_desc}_mask.png"
                
                # Save variant image and mask
                variant_path = self.synthetic_path / variant_name
                mask_path = self.masks_path / mask_name
                
                cv2.imwrite(str(variant_path), variant_img)
                cv2.imwrite(str(mask_path), variant_mask)
                
                total_generated += 1
            
            successful_images += 1
            print(f"✅ Generated {len(variants)} defective variants")
            
            # Progress update every 10 images
            if (img_idx + 1) % 10 == 0:
                print(f"\n📊 Progress: {img_idx + 1}/{len(image_files)} images processed")
                print(f"📊 Total variants generated so far: {total_generated}")
                
        except Exception as e:
            print(f"❌ Error processing {img_file.name}: {e}")
            continue
    
    # Final summary
    print(f"\n🎉 Defect Augmentation Complete!")
    print(f"📊 Successfully processed: {successful_images}/{len(image_files)} images")
    print(f"📊 Total defective variants generated: {total_generated}")
    print(f"📊 Average variants per image: {total_generated/successful_images:.1f}")
    
    # Save metadata
    metadata = {
        'original_images_processed': successful_images,
        'total_images_found': len(image_files),
        'variants_per_image_target': variants_per_image,
        'total_variants_generated': total_generated,
        'intensity_range': intensity_range,
        'defect_types': ['dirt', 'shadow', 'oxidation', 'flux', 'scratches'],
        'process_type': 'defect_augmentation'
    }
    
    with open(self.base_path / 'defect_augmentation_metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"💾 Metadata saved to: {self.base_path / 'defect_augmentation_metadata.pkl'}")
    
    return metadata


In [ ]:
# Example: Process entire dataset with rectangular defects only
def process_rectangular_defect_dataset(
    image_dir: str, 
    mask_dir: str,
    output_dir: str,
    variants_per_image: int = 15
    ):
    """
    Process dataset to generate rectangular defective variants
    
    🔧 STRICT REQUIREMENT: All defects are rectangular or rotated rectangular shapes
    
    Args:
        image_dir: Directory with original pin images
        mask_dir: Directory with corresponding masks  
        output_dir: Directory to save defective variants
        variants_per_image: Number of defective variants per original
    """
    
    print("🔧 RECTANGULAR DEFECT GENERATION SYSTEM")
    print("=" * 50)
    print("✅ Defect shapes: ONLY rectangles and rotated rectangles")
    print("✅ Defect types: dirt, shadow, oxidation, flux, scratches")
    print("✅ High intensity defects for clear visibility")
    print("✅ Descriptive filenames with defect information")
    
    # Initialize synthesizer
    synth = SemiconductorImageSynthesizer(base_path=Path(output_dir))
    
    # Process dataset
    metadata = synth.process_defect_augmentation_dataset(
        image_dir=image_dir,
        mask_dir=mask_dir,
        variants_per_image=variants_per_image,
        intensity_range=(0.5, 0.9)  # High intensity for visible defects
    )
    
    print("\n🎯 RECTANGULAR DEFECT GENERATION COMPLETE!")
    print(f"📊 Generated {metadata['total_variants_generated']} rectangular defective variants")
    print(f"📊 From {metadata['original_images_processed']} original pin images")
    
    return metadata

# Example usage (commented out - uncomment to run)
# rectangular_metadata = process_rectangular_defect_dataset(
#     image_dir=r'C:\path\to\original\pins',
#     mask_dir=r'C:\path\to\original\masks', 
#     output_dir=r'C:\path\to\rectangular\defects',
#     variants_per_image=15
# )


In [ ]:
# Visualization function to show rectangular defects
def visualize_rectangular_defects(original_img, original_mask, defective_variants):
    """
    Visualize original pin vs rectangular defective variants
    
    Shows clear evidence that all defects are rectangular/rotated rectangular
    """
    import matplotlib.pyplot as plt
    
    n_variants = min(6, len(defective_variants))  # Show up to 6 variants
    fig, axes = plt.subplots(2, n_variants + 1, figsize=(20, 8))
    
    # Show original
    axes[0, 0].imshow(original_img, cmap='gray')
    axes[0, 0].set_title('Original Pin', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[1, 0].imshow(original_mask, cmap='gray')
    axes[1, 0].set_title('Original Mask', fontsize=12, fontweight='bold') 
    axes[1, 0].axis('off')
    
    # Show rectangular defective variants
    for i in range(n_variants):
        variant_img, variant_mask, defect_desc = defective_variants[i]
        
        # Show defective image
        axes[0, i+1].imshow(variant_img, cmap='gray')
        axes[0, i+1].set_title(f'Rectangular Defect {i+1}', fontsize=10, fontweight='bold')
        axes[0, i+1].axis('off')
        
        # Show updated mask (highlighting rectangular defect areas)
        axes[1, i+1].imshow(variant_mask, cmap='gray')
        axes[1, i+1].set_title(f'Mask: {defect_desc[:20]}...', fontsize=8)
        axes[1, i+1].axis('off')
    
    plt.tight_layout()
    plt.suptitle('🔧 RECTANGULAR DEFECT GENERATION - All defects are rectangles/rotated rectangles', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Print defect details
    print("🔧 RECTANGULAR DEFECT DETAILS:")
    print("=" * 60)
    for i, (_, _, defect_desc) in enumerate(defective_variants[:n_variants]):
        print(f"Variant {i+1}: {defect_desc}")
    print("\n✅ ALL DEFECTS ARE GUARANTEED TO BE RECTANGULAR OR ROTATED RECTANGULAR SHAPES!")

# Example usage (commented out - uncomment when you have test data)
# if 'rectangular_variants' in locals():
#     visualize_rectangular_defects(test_img, test_mask, rectangular_variants)


In [ ]:
img_synth = SemiconductorImageSynthesizer(
    base_path=Path(r'C:\Users\goni\workspace\projects\data\easy_end_test\image_gen')
)

In [ ]:
process_rectangular_defect_dataset(
    image_dir=str(im_path),  # Your pin images directory
    mask_dir=str(msk_path),  # Your masks directory
    output_dir=str(output_path),
    variants_per_image=10,
)

🔧 RECTANGULAR DEFECT GENERATION SYSTEM
✅ Defect shapes: ONLY rectangles and rotated rectangles
✅ Defect types: dirt, shadow, oxidation, flux, scratches
✅ High intensity defects for clear visibility
✅ Descriptive filenames with defect information
🎯 Starting Defect Augmentation Process...
Target: 10 defective variants per original image
Defect intensity range: 0.5 - 0.9
Found 19 original images to process

📸 Processing image 1/19: VFV4.9.0.3_2025060418244066_Out_138_Missing Lead_image2.png
✅ Generated 10 defective variants

📸 Processing image 2/19: VFV4.9.0.3_2025060418324293_Out_138_Missing Lead_image2_2.png
✅ Generated 10 defective variants

📸 Processing image 3/19: VFV4.9.0.3_2025060419025109_Out_138_Missing Lead_image2.png
✅ Generated 10 defective variants

📸 Processing image 4/19: VFV4.9.0.3_2025060419241981_Out_138_Missing Lead_image2_1.png
✅ Generated 10 defective variants

📸 Processing image 5/19: VFV4.9.0.3_2025060419243230_Out_138_Missing Lead_image2.png
✅ Generated 10 defectiv

{'original_images_processed': 19,
 'total_images_found': 19,
 'variants_per_image_target': 10,
 'total_variants_generated': 190,
 'intensity_range': (0.5, 0.9),
 'defect_types': ['dirt', 'shadow', 'oxidation', 'flux', 'scratches'],
 'process_type': 'defect_augmentation'}

In [ ]:
#| export
@patch
def save_2channel_image(
    self:SemiconductorImageSynthesizer, 
    image: np.ndarray, # 2-channel image (grayscale + alpha)
    filepath: str # path to save the image
    ):
    """Save 2-channel image by converting to 4-channel RGBA"""
    if image.shape[2] != 2:
        raise ValueError(f"Expected 2-channel image, got {image.shape[2]} channels")
    
    # Convert 2-channel (gray + alpha) to 4-channel RGBA
    h, w = image.shape[:2]
    rgba_image = np.zeros((h, w, 4), dtype=np.uint8)
    
    # Set RGB channels to the grayscale value
    rgba_image[:, :, 0] = image[:, :, 0]  # R
    rgba_image[:, :, 1] = image[:, :, 0]  # G  
    rgba_image[:, :, 2] = image[:, :, 0]  # B
    rgba_image[:, :, 3] = image[:, :, 1]  # A (alpha)
    
    # Save as PNG to preserve alpha channel
    cv2.imwrite(filepath, rgba_image)


In [ ]:
#| export
@patch  
def load_2channel_image(
    self:SemiconductorImageSynthesizer,
    filepath: str # path to load the image from
    ) -> np.ndarray:
    """Load 4-channel RGBA image and convert back to 2-channel"""
    rgba_image = cv2.imread(filepath, cv2.IMREAD_UNCHANGED)
    
    if rgba_image is None:
        raise ValueError(f"Could not load image from {filepath}")
    
    if len(rgba_image.shape) == 3 and rgba_image.shape[2] == 4:
        # Convert RGBA back to 2-channel (grayscale + alpha)
        gray_alpha = np.zeros((rgba_image.shape[0], rgba_image.shape[1], 2), dtype=np.uint8)
        gray_alpha[:, :, 0] = rgba_image[:, :, 0]  # Use R channel as grayscale
        gray_alpha[:, :, 1] = rgba_image[:, :, 3]  # Use A channel as alpha
        return gray_alpha
    else:
        raise ValueError(f"Expected 4-channel RGBA image, got shape {rgba_image.shape}")


In [ ]:
#| export
@patch
def save_2channel_as_separate_files(
    self:SemiconductorImageSynthesizer,
    image: np.ndarray, # 2-channel image  
    base_filepath: str # base path without extension
    ):
    """Save 2-channel image as separate grayscale and mask files"""
    if image.shape[2] != 2:
        raise ValueError(f"Expected 2-channel image, got {image.shape[2]} channels")
    
    # Save grayscale channel
    gray_path = f"{base_filepath}_gray.png"
    cv2.imwrite(gray_path, image[:, :, 0])
    
    # Save alpha/mask channel  
    mask_path = f"{base_filepath}_mask.png"
    cv2.imwrite(mask_path, image[:, :, 1])
    
    return gray_path, mask_path


In [ ]:
#| export
@patch
def load_2channel_from_separate_files(
    self:SemiconductorImageSynthesizer,
    base_filepath: str # base path without extension
    ) -> np.ndarray:
    """Load 2-channel image from separate grayscale and mask files"""
    gray_path = f"{base_filepath}_gray.png"
    mask_path = f"{base_filepath}_mask.png"
    
    gray_img = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
    mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if gray_img is None or mask_img is None:
        raise ValueError(f"Could not load images from {gray_path} or {mask_path}")
    
    # Combine into 2-channel image
    combined = np.zeros((gray_img.shape[0], gray_img.shape[1], 2), dtype=np.uint8)
    combined[:, :, 0] = gray_img
    combined[:, :, 1] = mask_img
    
    return combined


In [ ]:
from platform import system
if system() == "Windows":
    data_path= Path(r'C:\Users\goni\workspace\projects\data\easy_end_test')
else:
    data_path= Path(r'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/')

im_path = Path(data_path, 'main_im2_cropped_sn_images_deduped')
msk_path  = Path(data_path, 'label_studio_masks')
print('Image path: ', im_path.exists())
print('Mask path: ', msk_path.exists())
images = list(Path(im_path).glob('*.png'))
image_names = get_name_(images)
print(f'Found {len(images)} images')


Image path:  True
Mask path:  True
Found 19 images


In [ ]:

foreground_bank = []
background_bank = []
for i in tqdm(image_names, total=len(image_names), desc='Processing images'):
    msk_fn = Path(msk_path, i)
    img_fn = Path(im_path, i)
    if msk_fn.exists():
        fg, bg = img_synth.extract_foreground_background(str(img_fn), str(msk_fn))
        if fg is not None and bg is not None:
            foreground_bank.append(fg)
            background_bank.append(bg)
            print(f'Processed {i}')
        else:
            print(f'No foreground or background found for {i}')
    else:
        print(f'Mask not found for {i}')
    print(f"Extracted {len(foreground_bank)} foreground and {len(background_bank)} background images")


 # Step 2: Deduplication
print("Step 2: Deduplicating images...")
fg_unique_indices = img_synth.deduplicate_images(foreground_bank)
bg_unique_indices = img_synth.deduplicate_images(background_bank)
        





Processing images:   0%|          | 0/19 [00:00<?, ?it/s]

Processed VFV4.9.0.3_2025060418244066_Out_138_Missing Lead_image2.png
Extracted 1 foreground and 1 background images
Processed VFV4.9.0.3_2025060418324293_Out_138_Missing Lead_image2_2.png
Extracted 2 foreground and 2 background images
Processed VFV4.9.0.3_2025060419025109_Out_138_Missing Lead_image2.png
Extracted 3 foreground and 3 background images
Processed VFV4.9.0.3_2025060419241981_Out_138_Missing Lead_image2_1.png
Extracted 4 foreground and 4 background images
Processed VFV4.9.0.3_2025060419243230_Out_138_Missing Lead_image2.png
Extracted 5 foreground and 5 background images
Processed VFV4.9.0.3_2025060419341510_Out_138_Missing Lead_image2_1.png
Extracted 6 foreground and 6 background images
Processed VFV4.9.0.3_2025060419400625_Out_138_Missing Lead_image2_3.png
Extracted 7 foreground and 7 background images
Processed VFV4.9.0.3_2025060419512628_Out_138_Missing Lead_image2.png
Extracted 8 foreground and 8 background images
Processed VFV4.9.0.3_2025060420005357_Out_138_Missing Le

In [ ]:
unique_foregrounds = [foreground_bank[i] for i in fg_unique_indices]
unique_backgrounds = [background_bank[i] for i in bg_unique_indices]











In [ ]:
# Save deduplicated images using helper methods for 2-channel images
for i, fg in enumerate(unique_foregrounds):
    # Save as RGBA (converts 2-channel to 4-channel RGBA for OpenCV compatibility)
    img_synth.save_2channel_image(fg, str(img_synth.fg_bank_path / f"fg_{i:04d}.png"))
        
for i, bg in enumerate(unique_backgrounds):
    # Save as RGBA (converts 2-channel to 4-channel RGBA for OpenCV compatibility)
    img_synth.save_2channel_image(bg, str(img_synth.bg_bank_path / f"bg_{i:04d}.png"))

print(f"Saved {len(unique_foregrounds)} foreground and {len(unique_backgrounds)} background images")

AttributeError: 'SemiconductorImageSynthesizer' object has no attribute 'save_2channel_image'

In [ ]:
#| hide
# Use nbdev_export without specifying filename to avoid path conflicts
import nbdev; nbdev.nbdev_export('55_image_gen.advanced_blending.ipynb')

ValueError: 'C:\\Users\\goni\\workspace\\projects\\nbs\\39_preprocessing.zero_degree_solder_pin.ipynb' is not in the subpath of 'C:\\Users\\goni\\workspace\\projects\\git_data\\new_test\\nbs'

In [ ]:
#| export
class SemiconductorPinToModuleCompositor:
    """
    🔬 Advanced Semiconductor Pin-to-Module Composition System
    
    Intelligently pastes pin images onto module backgrounds with:
    - Smart placement avoiding existing pins
    - Advanced blending for seamless integration  
    - Semiconductor-specific lighting and shadows
    - PCB substrate texture matching
    """
    
    def __init__(self, pin_spacing_threshold: int = 15):
        self.pin_spacing_threshold = pin_spacing_threshold
        self.placement_attempts = 100
        
    def find_valid_pin_locations(
        self, 
        module_image: np.ndarray,    # Background module image
        module_mask: np.ndarray,     # Existing pin locations (DON'T paste here)
        pin_mask: np.ndarray,        # Pin to be placed
        num_locations: int = 5       # Number of placement locations to find
        ) -> List[Tuple[int, int]]:
        """Find valid locations to place pins avoiding existing pin areas"""
        
        h_module, w_module = module_image.shape[:2]
        h_pin, w_pin = pin_mask.shape[:2]
        valid_locations = []
        
        # Create exclusion mask (existing pins + spacing buffer)
        exclusion_mask = module_mask.copy()
        
        # Dilate existing pin areas to enforce minimum spacing
        if np.any(exclusion_mask > 127):
            kernel = np.ones((self.pin_spacing_threshold, self.pin_spacing_threshold), np.uint8)
            exclusion_mask = cv2.dilate(exclusion_mask, kernel, iterations=1)
        
        # Find valid placement locations
        for attempt in range(self.placement_attempts):
            x = random.randint(0, max(0, w_module - w_pin))
            y = random.randint(0, max(0, h_module - h_pin))
            
            # Check if placement area is free from existing pins
            placement_area = exclusion_mask[y:y+h_pin, x:x+w_pin]
            
            if np.all(placement_area <= 127):  # No existing pins in this area
                # Prefer substrate areas (typically darker than traces)
                substrate_area = module_image[y:y+h_pin, x:x+w_pin]
                substrate_brightness = np.mean(substrate_area)
                
                if substrate_brightness < np.mean(module_image) * 1.3:
                    valid_locations.append((x, y))
                    
                    # Update exclusion mask for next placement
                    exclusion_mask[y:y+h_pin, x:x+w_pin] = 255
                    
                    if len(valid_locations) >= num_locations:
                        break
        
        return valid_locations


In [ ]:
#| export
class SemiconductorPinToModuleCompositor:
    """
    🔬 Advanced Semiconductor Pin-to-Module Composition System
    
    🎯 KEY FEATURES:
    - Smart pin placement avoiding existing pins (uses masks)
    - Advanced blending for seamless integration
    - Semiconductor-specific lighting and shadows
    - PCB substrate texture matching
    - Module background preservation (no changes to original)
    """
    
    def __init__(self, pin_spacing_threshold: int = 15):
        self.pin_spacing_threshold = pin_spacing_threshold
        self.placement_attempts = 100 # Number of attempts to find valid pin locations
        
        


In [ ]:
whole_img = SemiconductorPinToModuleCompositor(
    pin_spacing_threshold=1
)

In [ ]:
im_name = 'src_img_recipe_17_idx_0_gen_image_7_VFV4.9.0.3_2025031710275097_ID_00062046905819403042510_In_17_FRONT_Pass_image2_pin_19_rotation_7_angle_360_20250410192019466221_2_4.png'
pin_name = 'VFV4.9.0.3_2025060419241981_Out_138_Missing Lead_image2_1_enhanced_realistic_rect_dirt_v04.png'
module_im_path = Path(fr'C:\Users\goni\workspace\projects\data\easy_end_test\train_patch\images\{im_name}')
pin_im_path = Path(fr'C:\Users\goni\workspace\projects\data\easy_end_test\synthetic_images_test\images\{pin_name}')
module_msk_path = Path(module_im_path.parent.parent, 'masks', f'{im_name}') 
pin_msk_path = Path(pin_im_path.parent, 'masks', f'{pin_name}')
print(pin_msk_path)
try:
    print(f'module im exist ={module_im_path.exists()}')
    print(f'module msk exist ={module_msk_path.exists()}')
    print(f'pin im exist ={pin_im_path.exists()}')
    print(f'pin msk exist ={pin_msk_path.exists()}')
    module_img = read_img(module_im_path, gray=True)
    module_msk = read_img(module_msk_path, gray=True)
    pin_img = read_img(pin_im_path, gray=True)
    pin_msk = read_img(pin_msk_path, gray=True)
except Exception as e:
    print(f"Error reading image: {e}")
    print(module_im_path.exists())
    module_img = None
#valid_locs =    

C:\Users\goni\workspace\projects\data\easy_end_test\synthetic_images_test\images\masks\VFV4.9.0.3_2025060419241981_Out_138_Missing Lead_image2_1_enhanced_realistic_rect_dirt_v04.png
module im exist =True
module msk exist =True
pin im exist =True
pin msk exist =False


In [ ]:
#| export
@patch
def find_valid_pin_locations_efficient(
    self:SemiconductorPinToModuleCompositor, 
    module_image: np.ndarray,    # Background module image
    module_mask: np.ndarray,     # Existing pin locations (DON'T paste here)
    pin_mask: np.ndarray,        # Pin to be placed
    num_locations: int = 5       # Number of placement locations to find
    ) -> List[Tuple[int, int]]:
    """
    🚀 EFFICIENT pin placement using connected components analysis
    
    WHY THIS IS BETTER THAN TEMPLATE MATCHING:
    - Template matching: O(W*H*w*h) complexity
    - Connected components: O(W*H) complexity  
    - 10-100x faster for large images!
    
    APPROACH:
    1. Find all black (background) regions
    2. Check if each region is large enough for pin
    3. Sample valid positions within suitable regions
    """
        
    h_module, w_module = module_image.shape[:2]
    h_pin, w_pin = pin_mask.shape[:2]
    valid_locations = []
        
    # Create exclusion mask (existing pins + spacing buffer)
    exclusion_mask = module_mask.copy()
        
    # Dilate existing pin areas to enforce minimum spacing
    if np.any(exclusion_mask > 127):
        kernel = np.ones((self.pin_spacing_threshold, self.pin_spacing_threshold), np.uint8)
        exclusion_mask = cv2.dilate(exclusion_mask, kernel, iterations=1)
        
    # 🎯 EFFICIENT APPROACH: Find black areas larger than pin size
    # Find all black regions (background areas where we can place pins)
    black_mask = (exclusion_mask <= 127).astype(np.uint8)
    
    # Find connected components (black regions)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(black_mask, connectivity=8)
    
    pin_area = h_pin * w_pin
    
    # Check each black region
    for i in range(1, num_labels):  # Skip background label (0)
        if len(valid_locations) >= num_locations:
            break
            
        # Get region stats: [left, top, width, height, area]
        left, top, width, height, area = stats[i]
        
        # Check if this black region is large enough for our pin
        if area >= pin_area and width >= w_pin and height >= h_pin:
            # Sample positions within this region
            step_x = max(1, w_pin // 4)  # Adaptive step size
            step_y = max(1, h_pin // 4)
            
            for y in range(top, min(top + height - h_pin + 1, h_module - h_pin), step_y):
                for x in range(left, min(left + width - w_pin + 1, w_module - w_pin), step_x):
                    # Double-check this specific location is completely free
                    if np.all(exclusion_mask[y:y+h_pin, x:x+w_pin] <= 127):
                        # Prefer substrate areas (typically darker than traces)
                        substrate_area = module_image[y:y+h_pin, x:x+w_pin]
                        substrate_brightness = np.mean(substrate_area)
                        
                        if substrate_brightness < np.mean(module_image) * 1.3:
                            valid_locations.append((x, y))
                            
                            # Update exclusion mask for next placement
                            exclusion_mask[y:y+h_pin, x:x+w_pin] = 255
                            
                            if len(valid_locations) >= num_locations:
                                break
                if len(valid_locations) >= num_locations:
                    break
    
    return valid_locations


In [ ]:
# Test the new efficient method
efficient_locations = whole_img.find_valid_pin_locations_efficient(
    module_image=module_img, 
    module_mask=module_msk, 
    pin_mask=pin_msk, 
    num_locations=3
)
x, y = efficient_locations[0]
x, y


(66, 0)

In [ ]:
pin_img

In [ ]:
    
    def match_substrate_properties(
        self,
        pin_image: np.ndarray,
        module_image: np.ndarray,
        placement_x: int,
        placement_y: int,
        pin_mask: np.ndarray
        ) -> np.ndarray:
        """Match pin background to local module substrate properties"""
        
        h_pin, w_pin = pin_image.shape[:2]
        
        # Extract local module area around placement
        local_module = module_image[placement_y:placement_y+h_pin, placement_x:placement_x+w_pin]
        
        if local_module.shape != pin_image.shape:
            return pin_image
        
        # Get background areas (non-pin areas) from both images
        pin_bg_mask = pin_mask <= 127
        
        if not np.any(pin_bg_mask):
            return pin_image
        
        # Match background brightness to local module substrate
        pin_bg_brightness = np.mean(pin_image[pin_bg_mask])
        module_bg_brightness = np.mean(local_module[pin_bg_mask])
        
        brightness_ratio = module_bg_brightness / (pin_bg_brightness + 1e-6)
        
        # Apply brightness matching
        matched_pin = pin_image.copy().astype(np.float32)
        matched_pin[pin_bg_mask] *= brightness_ratio
        
        # Add local substrate texture to pin background
        substrate_texture = local_module[pin_bg_mask] - module_bg_brightness
        texture_strength = 0.2  # Subtle texture blending
        matched_pin[pin_bg_mask] += substrate_texture * texture_strength
        
        return np.clip(matched_pin, 0, 255).astype(np.uint8)
    
    def apply_pin_shadows(
        self,
        composite_image: np.ndarray,
        pin_mask: np.ndarray,
        placement_x: int,
        placement_y: int,
        shadow_intensity: float = 0.25
        ) -> np.ndarray:
        """Apply realistic pin shadows on module substrate"""
        
        result = composite_image.copy().astype(np.float32)
        h_pin, w_pin = pin_mask.shape[:2]
        
        # Shadow offset (typical inspection lighting from top-left)
        shadow_offset_x = 3
        shadow_offset_y = 3
        
        # Calculate shadow boundaries
        shadow_start_y = max(0, placement_y + shadow_offset_y)
        shadow_end_y = min(composite_image.shape[0], placement_y + h_pin + shadow_offset_y)
        shadow_start_x = max(0, placement_x + shadow_offset_x)
        shadow_end_x = min(composite_image.shape[1], placement_x + w_pin + shadow_offset_x)
        
        # Calculate corresponding pin mask boundaries
        pin_start_y = max(0, -shadow_offset_y)
        pin_end_y = pin_start_y + (shadow_end_y - shadow_start_y)
        pin_start_x = max(0, -shadow_offset_x)
        pin_end_x = pin_start_x + (shadow_end_x - shadow_start_x)
        
        if (shadow_end_y > shadow_start_y and shadow_end_x > shadow_start_x and
            pin_end_y <= h_pin and pin_end_x <= w_pin):
            
            # Create shadow mask
            shadow_mask = np.zeros((shadow_end_y - shadow_start_y, shadow_end_x - shadow_start_x), dtype=np.uint8)
            pin_shadow_area = pin_mask[pin_start_y:pin_end_y, pin_start_x:pin_end_x]
            shadow_mask = pin_shadow_area.copy()
            
            # Blur shadow for soft edges
            shadow_mask = cv2.GaussianBlur(shadow_mask, (7, 7), 2.0)
            
            # Apply shadow effect
            shadow_areas = shadow_mask > 50
            if np.any(shadow_areas):
                shadow_strength = (shadow_mask[shadow_areas] / 255.0) * shadow_intensity
                shadow_coords = np.where(shadow_areas)
                global_y = shadow_coords[0] + shadow_start_y
                global_x = shadow_coords[1] + shadow_start_x
                result[global_y, global_x] *= (1 - shadow_strength)
        
        return np.clip(result, 0, 255).astype(np.uint8)


In [ ]:
# Complete Semiconductor Pin-to-Module Composition Methods


In [ ]:
#| export
@patch
def advanced_semiconductor_blending(
    self:SemiconductorPinToModuleCompositor,
    pin_image: np.ndarray,
    module_image: np.ndarray,
    pin_mask: np.ndarray,
    placement_x: int,
    placement_y: int
    ) -> Tuple[np.ndarray, np.ndarray]:
    """
    🔬 Advanced blending for semiconductor pin-to-module composition
    
    BLENDING PIPELINE:
    1. Substrate property matching
    2. Multi-scale Laplacian blending
    3. Realistic shadow integration
    4. Edge refinement
    """
    
    h_pin, w_pin = pin_image.shape[:2]
    h_module, w_module = module_image.shape[:2]
    
    # Ensure placement is within bounds
    if (placement_x + w_pin > w_module or placement_y + h_pin > h_module):
        return module_image, np.zeros_like(module_image)
    
    # Step 1: Match pin to local substrate properties
    matched_pin = self.match_substrate_properties(
        pin_image, module_image, placement_x, placement_y, pin_mask
    )
    
    # Step 2: Create composite
    composite = module_image.copy().astype(np.float32)
    composite_mask = np.zeros_like(module_image, dtype=np.uint8)
    
    # Extract placement area from module
    module_area = module_image[placement_y:placement_y+h_pin, placement_x:placement_x+w_pin]
    
    # Step 3: Advanced blending
    blended_area = self.laplacian_blend_pins(matched_pin, module_area, pin_mask)
    
    # Step 4: Place blended pin into composite
    composite[placement_y:placement_y+h_pin, placement_x:placement_x+w_pin] = blended_area
    
    # Step 5: Update composite mask
    composite_mask[placement_y:placement_y+h_pin, placement_x:placement_x+w_pin] = pin_mask
    
    # Step 6: Apply realistic pin shadows
    composite = self.apply_pin_shadows(composite, pin_mask, placement_x, placement_y)
    
    return composite.astype(np.uint8), composite_mask
